# GSWA HyLogger — Step 0: *What data do we actually have?*

**Goal of this notebook:** open every piece of data we were given, look at it, and write down
in plain words what it is. No modelling, no clustering, no pipeline yet. Just *understanding*.

### The one-paragraph version of the project
Mining companies drill holes and pull up long rods of rock called **core**. The Geological Survey
of WA (GSWA) runs each rod under a scanner (the **HyLogger**) that shines light on the rock every
few centimetres and records how much light bounces back at each colour/wavelength. Different
minerals absorb light at different wavelengths, so that "light fingerprint" tells you which
minerals are present at every depth. There are ~2000 scanned holes across WA. Nobody has labelled
them. Our job is to build a tool that groups holes by their mineral fingerprint, flags odd depths,
and lets a geoscientist find follow-up targets in minutes instead of months.

### What this notebook covers
| # | Section | What you'll learn |
|---|---------|-------------------|
| 1 | Glossary | What all the jargon means (loaded from GSWA's own vocabulary file) |
| 2 | The WA-wide catalog | How many holes exist, how big, which are open vs confidential |
| 3 | The live API (`nvcl-kit`) | Getting hole locations and data **without** downloading 2.3 TB |
| 4 | One hole, opened up | The anatomy of a single HyLogger package |
| 5 | Where each reading sits | Tray / section / depth — how a row maps to a place in the rock |
| 6 | The spectra | The raw light measurements themselves |
| 7 | The scalars | The 150 pre-computed columns, including the mineral answers |
| 8 | Mineralogy downhole | The "strip log" — minerals plotted against depth |
| 9 | Data quality | What's masked, invalid, noisy, or missing |
| 10 | Core photos | The actual pictures of the rock |
| 11 | Tidy export | One clean table per hole, saved to CSV |
| 12 | What we learned | Findings + open questions for the client |

### The two libraries the client recommended
- **`pytsg`** — reads the `.tsg` / `.bip` files that hold the spectra. Used from section 4 on.
- **`nvcl-kit`** — a Python wrapper around GSWA's live NVCL web services. Used in section 3.

Both are from GSWA / AuScope themselves, so they define the "supported" way in.

## Setup

Run this cell first. It finds the data folders automatically and checks the libraries are there.

If anything is missing, run this once in a terminal:

```
pip install pytsg nvcl-kit pandas matplotlib openpyxl pillow
```

In [ ]:
from pathlib import Path
import io
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pytsg import parse_tsg

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ---------------------------------------------------------------
# WHERE THINGS LIVE
# ---------------------------------------------------------------
# This notebook lives in <project>/data_view/, so the project root is one level up.
# If you move the notebook, just set ROOT by hand instead.
try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd().parent if Path.cwd().name == "data_view" else Path.cwd()

OUT = ROOT / "data_view" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

# The email folder that holds the two spreadsheets + README.
DOC_DIR = next(
    (p for p in ROOT.iterdir() if p.is_dir() and p.name.startswith("RE_ GSWA HyLogger")),
    None,
)

# A HyLogger package = a folder containing a "*_tsg.tsg" file.
# We search the whole project so it keeps working when you download more holes.
PACKAGES = sorted({p.parent for p in ROOT.rglob("*_tsg.tsg")})

print("Project root :", ROOT)
print("Outputs      :", OUT)
print("Docs folder  :", DOC_DIR.name if DOC_DIR else "NOT FOUND")
print(f"HyLogger packages found: {len(PACKAGES)}")
for p in PACKAGES:
    print("   -", p.relative_to(ROOT))

In [ ]:
def read_bytes_shared(path):
    """Read a file on Windows even while Excel has it open.

    Excel holds an open handle on a spreadsheet, and Python's built-in open() asks for a
    share mode that clashes with it — you get "PermissionError: Permission denied" even
    though you only want to read. Asking Windows directly, with all three share flags,
    works. This is only needed on Windows; elsewhere we just read the file normally.
    """
    if os.name != "nt":
        return Path(path).read_bytes()

    import ctypes
    from ctypes import wintypes

    GENERIC_READ = 0x80000000
    SHARE_READ_WRITE_DELETE = 0x01 | 0x02 | 0x04
    OPEN_EXISTING = 3
    INVALID_HANDLE = ctypes.c_void_p(-1).value

    create_file = ctypes.windll.kernel32.CreateFileW
    create_file.restype = wintypes.HANDLE
    handle = create_file(str(path), GENERIC_READ, SHARE_READ_WRITE_DELETE,
                         None, OPEN_EXISTING, 0, None)
    if handle == INVALID_HANDLE:
        raise OSError(f"Could not open {path}")

    try:
        chunks = []
        buffer = ctypes.create_string_buffer(1 << 20)
        bytes_read = wintypes.DWORD()
        while True:
            ok = ctypes.windll.kernel32.ReadFile(
                handle, buffer, len(buffer), ctypes.byref(bytes_read), None
            )
            if not ok:
                raise OSError(f"Failed reading {path}")
            if bytes_read.value == 0:
                break
            chunks.append(buffer.raw[: bytes_read.value])
        return b"".join(chunks)
    finally:
        ctypes.windll.kernel32.CloseHandle(handle)


def read_excel_safely(path, **kwargs):
    """Read an Excel file even if it is currently open in Excel or sitting in OneDrive."""
    try:
        return pd.read_excel(path, **kwargs)
    except PermissionError:
        print(f"   ({Path(path).name} is open in another program — reading it anyway)")
        return pd.read_excel(io.BytesIO(read_bytes_shared(path)), **kwargs)


# TSG writes "no value here" as the smallest possible 32-bit float, not as NaN.
# If you forget this, every average and every plot is destroyed by -3.4e38.
TSG_NULL_THRESHOLD = -1e38


def clean_tsg_nulls(df):
    """Return a copy of df with TSG's sentinel values turned into proper NaN."""
    out = df.copy()
    for col in out.columns:
        s = out[col]
        if pd.api.types.is_numeric_dtype(s):
            out[col] = s.where(s > TSG_NULL_THRESHOLD)
        else:
            out[col] = s.replace("", np.nan)
    return out


print("Helpers ready.")

---
## 1. Glossary — decoding the jargon

GSWA maintains a controlled vocabulary of HyLogger terminology. This is genuinely the most
useful file we were given: it is the client telling us, in their own words, what every
abbreviation means. Load it and keep it beside you.

In [ ]:
vocab = pd.DataFrame()
if DOC_DIR is not None:
    vocab_path = next(DOC_DIR.glob("Hylogger_vocab*.xlsx"), None)
    if vocab_path:
        # Row 0 of the sheet is instructions for whoever fills it in, so we skip it.
        vocab = read_excel_safely(vocab_path, sheet_name="Concepts", header=1).iloc[1:]
        vocab = vocab.rename(
            columns={
                "Concept IRI*": "id",
                "Preferred Label*": "term",
                "Definition*": "definition",
                "Alternate Label": "aka",
                "Broader Concept IRI": "parent",
            }
        )[["id", "term", "definition", "aka", "parent"]]
        vocab = vocab.dropna(subset=["term"]).reset_index(drop=True)

print(f"{len(vocab)} terms defined by GSWA.\n")
print("Grouped under these parent concepts:")
print(vocab["parent"].value_counts().to_string())

In [ ]:
def explain(query):
    """Look a term up in GSWA's vocabulary. Try explain('TSA') or explain('albedo')."""
    if vocab.empty:
        print("Vocabulary not loaded.")
        return
    q = str(query).lower()
    hits = vocab[
        vocab["term"].str.lower().str.contains(q, na=False)
        | vocab["aka"].astype(str).str.lower().str.contains(q, na=False)
        | vocab["id"].astype(str).str.lower().str.contains(q, na=False)
    ]
    if hits.empty:
        print(f"No vocabulary entry matching '{query}'.")
        return
    for _, r in hits.iterrows():
        print(f"\n{'=' * 78}\n{r['term']}")
        if isinstance(r["aka"], str):
            print(f"  also known as: {r['aka']}")
        print(f"{'-' * 78}\n{r['definition']}")


# The terms you will meet constantly in the scalar table:
for term in ["TSA", "Grp", "Wt", "Albedo", "Mask"]:
    explain(term)

# Worth noting: the vocabulary does NOT define every column name we actually meet.
print("\n" + "=" * 78)
for missing in ["sTSAS", "uTSAS", "sTSAV"]:
    found = not vocab.empty and vocab["term"].str.lower().eq(missing.lower()).any()
    print(f"Is '{missing}' (a real column name in the data) defined in the vocabulary? {found}")

Note the file is called `Hylogger_vocab_working_...` — it is a **work in progress**. It defines
`sTSA+` and `sTSAV` but not `sTSAS`, even though `sTSAS` is a column name in the actual data.
Small thing, but exactly the kind of gap to feed back to GSWA: we're a useful test of their
vocabulary's completeness.

**The short version of the abbreviations, since you'll see them everywhere:**

- **TSA** = *The Spectral Assistant* — the software's automatic mineral identification. It
  compares each measured spectrum against a reference library and reports the best matches.
- **s / u prefix** = which reference library was used: **s**ystem (GSWA's standard) or
  **u**ser (customised). `sTSAS` = system library, SWIR range. `uTSAS` = user library, SWIR.
  `sTSAV` = system library, VNIR range.
- **Min1 / Min2 / Min3** = 1st, 2nd, 3rd most abundant mineral at that reading.
- **Wt1 / Wt2 / Wt3** = how much of each ("spectral weight" — roughly abundance, 0–1).
- **Grp1 / Grp2 / Grp3** = the mineral *family* of Min1/Min2/Min3 (e.g. `CHLORITE`, `WHITE-MICA`).
- **Error** = how badly the match fitted. High error = don't trust the mineral name.
- **Albedo** = overall brightness of the rock. Dark rock reflects less light.

**The mental model:** `Min1 sTSAS` is the software's *guess*, not ground truth. Nobody
has verified it. That is exactly why the capstone brief says the patterns must be
"statistically defensible and explainable against the logged geology".

---
## 2. The WA-wide catalog — how much data exists in total

`NVCL_WA_Catalog.xlsx` is the index of everything on the NCI THREDDS server
(https://thredds.nci.org.au/thredds/catalog/rs07/WA/catalog.html). One row = one scanned
drill hole, downloadable as a zip. This tells us the size of the problem before we download
anything.

In [ ]:
catalog = pd.DataFrame()
if DOC_DIR is not None:
    cat_path = next(DOC_DIR.glob("NVCL_WA_Catalog*.xlsx"), None)
    if cat_path:
        catalog = read_excel_safely(cat_path, sheet_name="catalog", header=1)
        catalog.columns = [
            "file", "size_text", "last_modified", "hole_name", "sql_snippet", "source",
            "anumber", "collar_id", "borehole_id", "wamex_conf", "wapims_conf",
        ]

        def size_to_mb(text):
            """'810.7 Mbytes' -> 810.7 ; '2.979 Gbytes' -> 3050.5"""
            try:
                value, unit = str(text).split()
                value = float(value)
                return value * 1024 if unit.lower().startswith("g") else value
            except (ValueError, AttributeError):
                return np.nan

        catalog["size_mb"] = catalog["size_text"].map(size_to_mb)
        catalog["is_cuttings"] = catalog["file"].str.contains("_cuttings", na=False)

print(f"Rows (= scanned holes) : {len(catalog)}")
print(f"Unique hole names      : {catalog['hole_name'].nunique()}")
print(f"Total download size    : {catalog['size_mb'].sum() / 1024:.0f} GB "
      f"({catalog['size_mb'].sum() / 1024 / 1024:.1f} TB)")
print(f"Median hole size       : {catalog['size_mb'].median():.0f} MB")
print(f"Biggest hole           : {catalog['size_mb'].max() / 1024:.1f} GB")
print(f"Unique WAMEX A-numbers : {catalog['anumber'].nunique()}   "
      f"(one report can cover many holes)")
print(f"Drill core vs cuttings : {(~catalog['is_cuttings']).sum()} core / "
      f"{catalog['is_cuttings'].sum()} cuttings")
print("\nAccess status (WAMEX column):")
print(catalog["wamex_conf"].value_counts(dropna=False).to_string())
print("\nAccess status (WAPIMS column, petroleum wells):")
print(catalog["wapims_conf"].value_counts(dropna=False).to_string())
catalog.head()

### Reading that

- **~2000 holes, ~2.3 TB.** We cannot download it all onto a laptop. The pipeline has to
  process hole-by-hole and keep only a small feature summary per hole. That is an
  architecture decision this table just made for us.
- **The size spread is huge** (2 MB to 14 GB). Most of a package's size is core photos,
  not spectra — so a "spectra-only" download is far cheaper if THREDDS allows it.
- **~600 rows have no A-number and no confidentiality flag.** Those are unresolved
  provenance — a real data-quality question for the client.
- **132 marked CONFIDENTIAL.** Worth confirming with GSWA whether those are in scope.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(catalog["size_mb"].dropna() / 1024, bins=60, color="#4C72B0")
axes[0].set_xlabel("package size (GB)")
axes[0].set_ylabel("number of holes")
axes[0].set_title("Most holes are small, a few are enormous")
axes[0].set_yscale("log")

access = catalog["wamex_conf"].fillna("(not stated)").value_counts()
axes[1].bar(access.index.astype(str), access.values, color=["#55A868", "#C44E52", "#8172B2"])
axes[1].set_title("WAMEX access status")
axes[1].set_ylabel("holes")
axes[1].tick_params(axis="x", rotation=20)

years = pd.to_datetime(catalog["last_modified"], errors="coerce").dt.year.value_counts().sort_index()
axes[2].bar(years.index.astype(int).astype(str), years.values, color="#DD8452")
axes[2].set_title("When each package was last published")
axes[2].set_ylabel("holes")

plt.tight_layout()
plt.show()

---
## 3. The live API — `nvcl-kit`

The catalog above describes 2.3 TB of zip files. But GSWA also runs **live web services**
that let you query holes and pull individual logs over HTTP, without downloading anything.
`nvcl-kit` (recommended by Tasman) wraps them.

This is a much better front door than the zips, and it changes our architecture: the pipeline
can pull what it needs per hole instead of hoarding terabytes.

**These cells need internet.** Set `RUN_NETWORK = False` to skip them if you're offline.

In [ ]:
RUN_NETWORK = True

boreholes = pd.DataFrame()

if RUN_NETWORK:
    from nvcl_kit.reader import NVCLReader
    from nvcl_kit.param_builder import param_builder

    # param_builder knows each state's service URLs. 'wa' = Geological Survey of WA.
    # max_boreholes=0 means "all of them".
    params = param_builder("wa", max_boreholes=0)
    print("Connecting to:", params.NVCL_URL)

    reader = NVCLReader(params)
    print("Connection OK:", reader.wfs is not None, "| error:", reader.wfs_error)

    borehole_objects = reader.get_boreholes_list()
    boreholes = pd.DataFrame([vars(b) for b in borehole_objects])
    boreholes["x"] = pd.to_numeric(boreholes["x"], errors="coerce")          # longitude
    boreholes["y"] = pd.to_numeric(boreholes["y"], errors="coerce")          # latitude
    boreholes["length_m"] = pd.to_numeric(boreholes["boreholeLength_m"], errors="coerce")

    print(f"\n{len(boreholes)} boreholes returned, with coordinates for "
          f"{boreholes['x'].notna().sum()}")
    print(f"Columns available: {list(boreholes.columns)}")

boreholes[["nvcl_id", "x", "y", "length_m", "drillingMethod", "operator", "project"]].head()

**Stop and appreciate what just happened.** In about a second, with no download, we got every
WA hole *with its latitude and longitude*, its length, who drilled it, the drilling method,
and the project name.

The capstone brief asks for a "WA map" front-end and "PostGIS for the geography axis".
**This table is that map layer.** We were about to write down "how do we get collar
coordinates?" as an open question — it's answered.

In [ ]:
if not boreholes.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    scatter = axes[0].scatter(
        boreholes["x"], boreholes["y"],
        c=boreholes["length_m"], cmap="viridis",
        s=14, alpha=0.75, vmin=0, vmax=boreholes["length_m"].quantile(0.95),
    )
    axes[0].set_xlabel("longitude")
    axes[0].set_ylabel("latitude")
    axes[0].set_title(f"All {len(boreholes)} HyLogger-scanned holes in WA")
    plt.colorbar(scatter, ax=axes[0], label="hole length (m)")

    methods = boreholes["drillingMethod"].value_counts().head(8)
    axes[1].barh(methods.index[::-1], methods.values[::-1], color="#4C72B0")
    axes[1].set_xlabel("holes")
    axes[1].set_title("Drilling method")

    plt.tight_layout()
    plt.show()

    print(f"Hole length: median {boreholes['length_m'].median():.0f} m, "
          f"max {boreholes['length_m'].max():.0f} m")
    print(f"Distinct projects: {boreholes['project'].nunique()}   "
          f"Distinct operators: {boreholes['operator'].nunique()}")

The clustering visible on that map is not random — holes bunch along known mineralised belts
(the Yilgarn goldfields, the Pilbara). "Where across WA do unusual mineral signatures cluster"
is the client's actual question, and this is the canvas it gets answered on.

In [ ]:
# Does the live API agree with the spreadsheet catalog? Never assume two sources match.
if not boreholes.empty and not catalog.empty:
    api_ids = set(boreholes["nvcl_id"].astype(str))
    catalog_ids = set(catalog["hole_name"].astype(str))

    print(f"In the API only        : {len(api_ids - catalog_ids):>5}")
    print(f"In the catalog only    : {len(catalog_ids - api_ids):>5}")
    print(f"In both                : {len(api_ids & catalog_ids):>5}")
    print("\nExamples present only in the catalog spreadsheet:")
    print("  ", sorted(catalog_ids - api_ids)[:8])

**Neither source is a superset of the other** — a few hundred holes each way. The catalog-only
names include `_wedge` holes (re-drills branching off a parent hole) and `_cuttings` entries
(rock chips, not solid core). Some are probably naming mismatches rather than genuinely
missing holes.

Don't silently pick one. **Which list is authoritative** is a question for Tasman, and until
it's answered we should treat ~1,700 holes as the confidently-available set.

In [ ]:
# Each hole exposes datasets; each dataset exposes logs we can pull individually.
if RUN_NETWORK and not boreholes.empty:
    example_hole = "05GJD001"  # the hole we have downloaded, so we can compare

    dataset_ids = reader.get_datasetid_list(example_hole)
    datasets = reader.get_dataset_list(example_hole)
    print(f"{example_hole} has {len(dataset_ids)} dataset(s): {dataset_ids}")

    # The description field is XML holding the collar + survey information.
    import xml.etree.ElementTree as ET

    meta_xml = ET.fromstring(datasets[0].description)
    survey = {child.tag: child.text for child in meta_xml}
    print("\nCollar / survey metadata straight from the API:")
    for key, value in survey.items():
        print(f"   {key:16s} {value}")

    scalar_logs = reader.get_scalar_logs(dataset_ids[0])
    print(f"\n{len(scalar_logs)} scalar logs available for download individually:")
    for log in scalar_logs[:10]:
        print(f"   {log.log_name:20s} type={log.log_type}  id={log.log_id}")

**`Azimuth`, `Inclination`, `RL`, `Latitude`, `Longitude`, `Datum`** — that is exactly the
collar-and-survey information the README's drilling diagram says you need for "spatial
reference and desurveying". It comes free with the dataset record.

And the scalar logs can be fetched one at a time. If all we want is `Min1 uTSAS` down a
hole, that's a few hundred KB instead of a 780 MB zip.

In [ ]:
# Pull one mineral log over HTTP and compare it against our local file. Same numbers?
if RUN_NETWORK and not boreholes.empty:
    from io import StringIO

    target = next((l for l in scalar_logs if l.log_name == "Min1 uTSAS"), None)
    if target:
        raw = reader.get_scalar_data([target.log_id])
        api_log = pd.read_csv(StringIO(raw))
        api_log = api_log.replace("null", np.nan)

        print(f"Downloaded {len(api_log):,} rows over HTTP")
        print(api_log.head())
        print(f"\nDepth range from API   : {api_log['StartDepth'].min():.1f} – "
              f"{api_log['StartDepth'].max():.1f} m")
        print("Compare this against the local file's depth range in section 5 — "
              "they should agree.")

### What section 3 changes about the plan

| Question we had | Answer |
|---|---|
| How do we get 2.3 TB onto a laptop? | We don't. Query per hole over HTTP. |
| Where do map coordinates come from? | `get_boreholes_list()` — all 1,897, instantly. |
| Where does collar/survey come from? | The dataset description XML: azimuth, dip, RL, datum. |
| Do we need full packages to prototype? | No. Pull scalar logs; download packages only when we need raw spectra. |

The zips still matter — they're the only route to the **full spectral cube** — but they
should be the exception, not the default.

---
## 4. One hole, opened up

We have one full package downloaded: **05GJD001**. Everything below is that single hole.

**What the files in the package are:**

| File | What it is |
|------|-----------|
| `*_tsg.tsg` | Text header for the visible/near-infrared scan — dimensions, metadata |
| `*_tsg.bip` | The actual VNIR–SWIR numbers (binary) |
| `*_tsg_tir.tsg` / `.bip` | Same pair for the thermal infrared scan |
| `*_tsg_cras.bip` | The high-resolution linescan colour imagery of the core (373 MB) |
| `*_tsg_hires.dat` | Laser profilometry — the 3D surface height of the core |
| `*_tsg.cal` | Calibration data |
| `*_tsg*.ini` | Software settings used during scanning |
| `*_0001.jpg …` | Photographs of each core tray |
| `*_Mosaic.jpg` | All trays stitched into one long image |

`pytsg.read_package()` reads the `.tsg`/`.bip` pairs and hands back tidy arrays.
We skip the 373 MB linescan (`read_cras_file=False`) — it's slow and we don't need it yet.

In [ ]:
PACKAGE_DIR = PACKAGES[0]  # change the index to look at a different hole
print("Reading:", PACKAGE_DIR.name)

package = parse_tsg.read_package(PACKAGE_DIR, read_cras_file=False)

nir = package.nir  # visible + near/short-wave infrared
tir = package.tir  # thermal infrared
lidar = package.lidar

# pytsg returns the bare class (not data) when a section is absent — HyLogger-3 has no MIR.
has_mir = isinstance(package.mir, parse_tsg.Spectra)

print("\nSections present:")
for name, section in [("nir (VNIR+SWIR)", nir), ("tir (thermal IR)", tir)]:
    print(f"  {name:20s} spectra {section.spectra.shape}   "
          f"{section.wavelength.min():.0f}–{section.wavelength.max():.0f} nm   "
          f"scalars {section.scalars.shape}")
print(f"  {'mir':20s} {'present' if has_mir else 'not present (HyLogger-3 has no MIR sensor)'}")
print(f"  {'lidar':20s} {lidar.shape} height readings")

print(f"\nEach section holds: {list(vars(nir).keys())}")

**Read that carefully — it's the shape of our whole problem:**

- **25,938 rows.** Each row is one scan position along the core.
- **531 columns of NIR** (380–2500 nm, one reading every 4 nm) and **341 of TIR**
  (6000–14500 nm, every 25 nm). That's the raw "fingerprint".
- **151 scalar columns** — pre-computed answers derived from those spectra by TSG software.
- NIR and TIR have the **same 25,938 rows**, so they line up row-for-row. Convenient.

So one hole is a `25938 × 872` spectral matrix plus a `25938 × 151` table.
Times ~2000 holes. That is why the brief says "PCA + k-means baseline first".

In [ ]:
# What the scanner itself recorded about this hole (straight from the .tsg header).
header_text = (PACKAGE_DIR / f"{PACKAGE_DIR.parent.name}_tsg.tsg")
if not header_text.exists():
    header_text = next(PACKAGE_DIR.glob("*_tsg.tsg"))
print(header_text.read_text(errors="ignore")[:1400])

The `Executive Summary` line is a goldmine — it carries the hole's identity and location
metadata: GSWA UWI, the WAMEX report ID, the tectonic unit, map sheets, and which core
library the physical rock sits in. **That is our join key to everything else** (assays,
geology logs, collar coordinates). Worth parsing properly in the pipeline.

---
## 5. Where each reading sits in the rock

A row number means nothing on its own. `sampleheaders` tells us where that reading physically is.

| Column | Meaning |
|--------|---------|
| `T` | Tray number — the physical box the core sits in |
| `L` | Line — which row within the tray (core is laid out in several rows per tray) |
| `P` | Position along that line |
| `D` | **Depth down the hole, in metres** — the column that matters most |
| `X` | Distance along the scan in mm |
| `H` | Hole ID |

In [ ]:
meta = nir.sampleheaders.copy()
meta["D"] = pd.to_numeric(meta["D"], errors="coerce")
meta["X"] = pd.to_numeric(meta["X"], errors="coerce")
meta = meta.rename(columns={"T": "tray", "L": "line", "P": "position", "D": "depth_m",
                            "X": "dist_mm", "H": "hole_id"})

print(meta.head())
print(f"\nHole            : {meta['hole_id'].iloc[0]}")
print(f"Depth range     : {meta['depth_m'].min():.2f} m to {meta['depth_m'].max():.2f} m "
      f"({meta['depth_m'].max() - meta['depth_m'].min():.0f} m of core scanned)")
print(f"Trays           : {meta['tray'].nunique()}")
print(f"Readings        : {len(meta):,}")

spacing = np.diff(np.sort(meta["depth_m"].values))
spacing = spacing[spacing > 0]
print(f"\nSpacing between readings (cm):")
print(f"  median {np.median(spacing) * 100:.2f} | "
      f"5th pct {np.percentile(spacing, 5) * 100:.2f} | "
      f"95th pct {np.percentile(spacing, 95) * 100:.2f} | "
      f"max {spacing.max() * 100:.1f}")
print(f"\n=> roughly one reading every {np.median(spacing) * 1000:.0f} mm of rock.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(meta.index, meta["depth_m"], lw=0.8, color="#4C72B0")
axes[0].set_xlabel("row number")
axes[0].set_ylabel("depth (m)")
axes[0].invert_yaxis()
axes[0].set_title("Rows run in depth order, top to bottom")

axes[1].hist(spacing * 100, bins=80, range=(0, 5), color="#55A868")
axes[1].set_xlabel("gap to next reading (cm)")
axes[1].set_ylabel("count")
axes[1].set_title("Sampling is regular — with occasional jumps")
axes[1].set_yscale("log")

per_tray = meta.groupby("tray").size()
axes[2].bar(range(len(per_tray)), per_tray.values, color="#DD8452")
axes[2].set_xlabel("tray (in order)")
axes[2].set_ylabel("readings")
axes[2].set_title("Readings per core tray")

plt.tight_layout()
plt.show()

The occasional large gaps in the middle histogram are **core loss** — places where rock
was missing, crumbled, or the tray had a marker block. Those gaps are real geology-adjacent
information, not noise, and any depth-based join has to respect them.

---
## 6. The spectra — the raw measurement

This is the actual data the whole project rests on. For every one of the 25,938 positions,
the scanner recorded how much light bounced back at each wavelength. Minerals show up as
**dips (absorption features)** at characteristic wavelengths — e.g. white mica absorbs
near 2200 nm, chlorite near 2250 nm.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

for ax, section, label, unit_hint in [
    (axes[0], nir, "VNIR + SWIR", "visible light starts at 380 nm"),
    (axes[1], tir, "Thermal IR", "heat wavelengths"),
]:
    wl = section.wavelength
    spec = section.spectra
    mean = spec.mean(axis=0)
    p10, p90 = np.percentile(spec, [10, 90], axis=0)

    ax.fill_between(wl, p10, p90, alpha=0.25, color="#4C72B0", label="10th–90th percentile")
    ax.plot(wl, mean, color="#C44E52", lw=2, label="average spectrum")
    for i in np.linspace(0, len(spec) - 1, 4).astype(int):
        ax.plot(wl, spec[i], lw=0.7, alpha=0.6, color="grey")
    ax.set_xlabel("wavelength (nm)")
    ax.set_ylabel("reflectance")
    ax.set_title(f"{label} — {spec.shape[1]} bands  ({unit_hint})")
    ax.legend(fontsize=8)

# Mark the two diagnostic absorption wavelengths GSWA's own scalars use.
for x, name in [(2200, "2200 nm\nwhite mica"), (2250, "2250 nm\nchlorite")]:
    axes[0].axvline(x, color="black", ls="--", lw=0.8)
    axes[0].text(x, axes[0].get_ylim()[1] * 0.95, name, fontsize=7, ha="center", va="top")

plt.tight_layout()
plt.show()

print("Grey lines are 4 individual readings; red is the average of all 25,938.")
print("The dips are what mineral identification keys off.")

In [ ]:
# Every spectrum at once: depth down the page, wavelength across, colour = reflectance.
# Vertical stripes = a mineral feature that persists. Horizontal bands = a change in rock type.
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

order = np.argsort(meta["depth_m"].values)
n_bins = 500

for ax, section, label in [(axes[0], nir, "VNIR + SWIR"), (axes[1], tir, "Thermal IR")]:
    spec = section.spectra[order]
    # Average groups of rows together so the image is a manageable size.
    step = max(1, len(spec) // n_bins)
    binned = spec[: (len(spec) // step) * step].reshape(-1, step, spec.shape[1]).mean(axis=1)
    depths = meta["depth_m"].values[order]
    im = ax.imshow(
        binned,
        aspect="auto",
        cmap="viridis",
        extent=[section.wavelength.min(), section.wavelength.max(), depths.max(), depths.min()],
    )
    ax.set_xlabel("wavelength (nm)")
    ax.set_ylabel("depth (m)")
    ax.set_title(f"{label}: every reading, stacked by depth")
    ax.grid(False)
    plt.colorbar(im, ax=ax, label="reflectance")

plt.tight_layout()
plt.show()

**This single image is the project in one picture.** The horizontal banding is real
geology — the rock changes as you go deeper, and the spectral fingerprint changes with it.
Finding those bands automatically, across 2000 holes, and proving they're real, *is* the task.

---
## 7. The scalars — 151 pre-computed columns

TSG software has already derived a lot from those spectra. Not all 151 columns are useful:
they split into a few families. Let's sort them out and see how complete each one is.

**Critical gotcha:** TSG writes "missing" as `-3.402823e+38` (the smallest 32-bit float),
not as a blank. Text columns use `""`. If you don't clean these first, every statistic you
compute will be garbage. That is what `clean_tsg_nulls()` in the setup cell handles.

In [ ]:
scalars = clean_tsg_nulls(nir.scalars)

# The "Depth1..25 / Centre1..25 / Width1..25" block is raw feature-extraction output:
# for each reading, the 25 strongest absorption dips (how deep, where, how wide).
feature_cols = [c for c in scalars.columns
                if any(c.startswith(p) and c[len(p):].isdigit()
                       for p in ("Depth", "Centre", "Width"))]

families = {
    "Feature extraction (25 strongest dips)": feature_cols,
    "Position / identity": ["Depth (m)", "Tray", "Section", "HoleID", "Domain",
                            "SecDist (mm)", "SecSamp", "TraySamp", "Date", "RelDate"],
    "Mineral results — system SWIR library": [c for c in scalars.columns if c.endswith("sTSAS")],
    "Mineral results — user SWIR library": [c for c in scalars.columns if c.endswith("uTSAS")],
    "Mineral results — system VNIR library": [c for c in scalars.columns if c.endswith("sTSAV")],
    "Spectral indices": ["Albedo", "w2200", "w2250", "White Mica w2200", "Chlorite w2250",
                         "Ferric oxide abundance 25062014", "Hematite-goethite_distr",
                         "prof_min", "RelRange"],
    "Quality / masking": ["Flags", "Final Mask", "HyLogDiag", "sec_end_mask",
                          "recovery_rate", "FeatExMethod", "NumFeats"],
}
accounted = {c for cols in families.values() for c in cols}
families["Other / unclear"] = [c for c in scalars.columns if c not in accounted]

for name, cols in families.items():
    cols = [c for c in cols if c in scalars.columns]
    if not cols:
        continue
    filled = scalars[cols].notna().mean().mean() * 100
    print(f"\n{name}  —  {len(cols)} columns, {filled:.0f}% filled")
    print("   ", ", ".join(cols[:12]) + (" …" if len(cols) > 12 else ""))

In [ ]:
# How complete is each of the columns we actually care about?
interesting = [c for c in scalars.columns if c not in feature_cols]
coverage = (scalars[interesting].notna().mean() * 100).sort_values()
coverage = coverage[coverage < 100]  # the 100%-full ones aren't interesting here

fig, ax = plt.subplots(figsize=(9, 10))
ax.barh(coverage.index, coverage.values, color="#4C72B0")
ax.set_xlabel("% of the 25,938 readings that have a value")
ax.set_title("Column completeness — the mineral answers are NOT complete")
ax.axvline(50, color="red", ls="--", lw=1)
plt.tight_layout()
plt.show()

print("Every column below the red line is missing for more than half the hole.")

### The most important finding so far

The mineral columns are only **~60–70% populated**, and the missing readings split into two
quite different groups:

| | Share | What it is |
|---|---|---|
| `Final Mask = 0` | 28.6% | **Not rock.** The scanner was looking at a wooden depth block, an empty tray section, or the tray edge. GSWA's vocabulary defines the mask as exactly this. |
| Kept, but no mineral | 8.0% | **Real rock the software couldn't identify.** Too dark, too noisy, or no diagnostic absorption features. |

The distinction matters. The first group should be discarded — it isn't geology. The second
group *is* geology and its own kind of signal: "unidentifiable" clusters somewhere.

Either way, any "anomaly" we detect must be checked against the mask before we claim it's
real. Silently dropping both groups would bias results toward well-behaved rock.

---
## 8. Mineralogy downhole — the strip log

Now the part a geoscientist actually recognises: what minerals, at what depth.

`Grp1 sTSAS` is the mineral *family* of the dominant mineral; `Min1 sTSAS` is the specific
mineral; `Wt1 sTSAS` is how much of it.

In [ ]:
minerals = pd.DataFrame({
    "depth_m": meta["depth_m"].values,
    "tray": meta["tray"].values,
    "group_1": scalars["Grp1 sTSAS"].values,
    "mineral_1": scalars["Min1 sTSAS"].values,
    "weight_1": scalars["Wt1 sTSAS"].values,
    "mineral_2": scalars["Min2 sTSAS"].values,
    "weight_2": scalars["Wt2 sTSAS"].values,
    "error": scalars["Error sTSAS"].values,
    "albedo": scalars["Albedo"].values,
})

print(f"Readings with a mineral identified: {minerals['mineral_1'].notna().sum():,} "
      f"of {len(minerals):,} ({minerals['mineral_1'].notna().mean():.0%})\n")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

grp = minerals["group_1"].value_counts().head(12)
axes[0].barh(grp.index[::-1], grp.values[::-1], color="#55A868")
axes[0].set_xlabel("readings")
axes[0].set_title("Dominant mineral GROUP (Grp1 sTSAS)")

mins = minerals["mineral_1"].value_counts().head(15)
axes[1].barh(mins.index[::-1], mins.values[::-1], color="#4C72B0")
axes[1].set_xlabel("readings")
axes[1].set_title("Dominant MINERAL (Min1 sTSAS)")

plt.tight_layout()
plt.show()

Note `INVALID` and `Aspectral` sitting among the top results. Those are the software saying
*"I could not identify this"* — they are not minerals. Treating them as a cluster label would
be a classic silent mistake.

In [ ]:
# The strip log: for every 2 m of hole, what fraction of readings is each mineral group?
BIN_M = 2.0
valid = minerals.dropna(subset=["group_1"]).copy()
valid = valid[~valid["group_1"].isin(["INVALID", "NOTAROK"])]
valid["depth_bin"] = (valid["depth_m"] // BIN_M) * BIN_M

composition = (
    pd.crosstab(valid["depth_bin"], valid["group_1"], normalize="index")
    .reindex(columns=valid["group_1"].value_counts().index)
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 9),
                               gridspec_kw={"width_ratios": [3, 1]}, sharey=True)

left = np.zeros(len(composition))
colors = plt.cm.tab20(np.linspace(0, 1, composition.shape[1]))
for colour, group in zip(colors, composition.columns):
    ax1.barh(composition.index, composition[group], left=left, height=BIN_M,
             label=group, color=colour)
    left += composition[group].values
ax1.set_xlabel(f"proportion of readings in each {BIN_M:.0f} m interval")
ax1.set_ylabel("depth (m)")
ax1.set_xlim(0, 1)
ax1.invert_yaxis()
ax1.set_title(f"Mineral groups down hole {meta['hole_id'].iloc[0]}")
ax1.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
ax1.grid(False)

albedo_profile = minerals.groupby((minerals["depth_m"] // BIN_M) * BIN_M)["albedo"].mean()
ax2.plot(albedo_profile.values, albedo_profile.index, color="#333333", lw=1)
ax2.set_xlabel("mean albedo\n(rock brightness)")
ax2.set_title("Brightness")

plt.tight_layout()
plt.show()

**This is the deliverable, in prototype form.** The brief asks for "interactive hole profiles"
and "strip logs" — that's this plot, made interactive and served from a web app.

Look at how the composition changes with depth: those transitions are candidate geological
boundaries. A clustering model should rediscover them without being told.

In [ ]:
# Does a "low error" filter actually change the picture? It should — this is a sanity check
# on whether the software's confidence numbers mean anything.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(minerals["error"].dropna(), bins=80, color="#C44E52")
axes[0].set_xlabel("Error sTSAS (lower = better match)")
axes[0].set_ylabel("readings")
axes[0].set_title("Match quality across the hole")

for label, subset in [
    ("all identified", minerals.dropna(subset=["mineral_1"])),
    ("best-fitting half", minerals[minerals["error"] < minerals["error"].median()]),
]:
    top = subset["mineral_1"].value_counts(normalize=True).head(8)
    axes[1].plot(top.index, top.values, marker="o", label=label)
axes[1].set_ylabel("share of readings")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()
axes[1].set_title("Do the top minerals survive a quality filter?")

plt.tight_layout()
plt.show()

---
## 9. Data quality — what we can and can't trust

Before any modelling, know what fraction of the data is real measurement versus artefact.

In [ ]:
quality = pd.DataFrame({
    "diagnosis": nir.scalars["HyLogDiag"].values,     # scanner's own verdict per reading
    "final_mask": nir.scalars["Final Mask"].values,   # 1 = keep, 0 = masked out
    "snr": scalars["SNR sTSAS"].values,               # signal-to-noise
    "error": scalars["Error sTSAS"].values,
    "depth_m": meta["depth_m"].values,
})

print("HyLogDiag — the scanner's verdict on each reading:")
print(quality["diagnosis"].value_counts().to_string())
print("   ('Good' is self-explanatory. PZ / AL PZ / KT / VO are undocumented — they are NOT")
print("    defined in GSWA's vocabulary file. Ask Tasman what they mean.)")

print(f"\nFinal Mask: {(quality['final_mask'] == 1).sum():,} kept "
      f"({(quality['final_mask'] == 1).mean():.0%}), "
      f"{(quality['final_mask'] == 0).sum():,} masked out")

dupes = meta["depth_m"].duplicated().sum()
print(f"\nDuplicate depth values : {dupes:,}")
print(f"Missing depth values   : {meta['depth_m'].isna().sum():,}")
print(f"Depths out of order    : {(np.diff(meta['depth_m'].values) < 0).sum():,}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(quality["snr"].dropna(), bins=80, color="#4C72B0")
axes[0].set_xlabel("signal-to-noise ratio")
axes[0].set_title("Higher = cleaner spectrum")

masked_by_depth = quality.groupby((quality["depth_m"] // 5) * 5)["final_mask"].mean()
axes[1].plot(masked_by_depth.values, masked_by_depth.index, color="#C44E52")
axes[1].set_xlabel("fraction kept")
axes[1].set_ylabel("depth (m)")
axes[1].invert_yaxis()
axes[1].set_title("Is masking spread evenly down the hole?")

axes[2].plot(lidar[:2000], lw=0.6, color="#55A868")
axes[2].set_xlabel("reading (first 2000)")
axes[2].set_ylabel("height (mm)")
axes[2].set_title("Laser profilometry — core surface height")

plt.tight_layout()
plt.show()

The middle plot matters: if masking clusters at particular depths, then "no data" is itself
correlated with geology (broken, weathered, or friable zones). An anomaly detector could
easily learn to flag *missingness* instead of mineralogy. Worth watching.

### Depth is not strictly increasing — and that's expected

The check above finds ~1,900 duplicate depths and ~1,500 places where depth goes *backwards*
from one row to the next. That is not corruption. Core is laid out in **several parallel rows
per tray**, and the scanner sweeps each row in turn — so at the end of every row the depth
jumps back a little before continuing. The `line` and `position` columns tell you where in
that zig-zag you are.

**The consequence is real though:** never assume `row order == depth order`. Any depth-based
join, rolling window, or resample must sort by depth first, and must decide how to combine
readings that share a depth. This is the kind of thing that silently corrupts a pipeline.

---
## 10. The core photos

Each package includes photographs of every tray plus a stitched mosaic. These are what
the geoscientist compares against when checking whether our clusters make sense.

In [ ]:
from PIL import Image

tray_photos = sorted(p for p in PACKAGE_DIR.glob("*.jpg") if "Mosaic" not in p.name)
mosaic = next(PACKAGE_DIR.glob("*Mosaic.jpg"), None)

print(f"Tray photos: {len(tray_photos)}  |  Mosaic: {'yes' if mosaic else 'no'}")

if tray_photos:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    for ax, photo in zip(axes, tray_photos[:2]):
        img = Image.open(photo)
        original_size = img.size
        img.thumbnail((1600, 1600))  # downscale so the notebook stays light
        ax.imshow(img)
        ax.set_title(f"{photo.name}   (full size {original_size[0]}×{original_size[1]} px)")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

You can see the wooden depth blocks and the breaks in the core. Those blocks are exactly
what the `HyLogDiag` flags and the mask are removing from the spectra.

---
## 11. Tidy export — one clean table per hole

Everything above lives in awkward nested objects. The pipeline wants one flat row per
reading. This is the shape that goes into a database and into a model.

In [ ]:
tidy = pd.DataFrame({
    "hole_id": meta["hole_id"].values,
    "sample_index": meta["sample"].values,
    "depth_m": meta["depth_m"].values,
    "tray": meta["tray"].values,
    "line": meta["line"].values,
    "position": meta["position"].values,
    "section": scalars["Section"].values,
    "keep": nir.scalars["Final Mask"].values == 1,
    "diagnosis": nir.scalars["HyLogDiag"].values,
    "albedo": scalars["Albedo"].values,
    "snr": scalars["SNR sTSAS"].values,
    "error": scalars["Error sTSAS"].values,
    "group_1": scalars["Grp1 sTSAS"].values,
    "mineral_1": scalars["Min1 sTSAS"].values,
    "weight_1": scalars["Wt1 sTSAS"].values,
    "group_2": scalars["Grp2 sTSAS"].values,
    "mineral_2": scalars["Min2 sTSAS"].values,
    "weight_2": scalars["Wt2 sTSAS"].values,
    "bound_water": scalars["Bound_Water sTSAS"].values,
    "unbound_water": scalars["Unbound_Water sTSAS"].values,
    "w2200": scalars["w2200"].values,
    "w2250": scalars["w2250"].values,
    "lidar_height": lidar,
})

hole = tidy["hole_id"].iloc[0]
tidy_path = OUT / f"{hole}_samples.csv"
tidy.to_csv(tidy_path, index=False)

print(f"Saved {len(tidy):,} rows x {tidy.shape[1]} columns")
print(f"  -> {tidy_path}")
tidy.head()

In [ ]:
# And the spectra as a plain numeric matrix — the input any model will actually consume.
spectra_path = OUT / f"{hole}_spectra.npz"
np.savez_compressed(
    spectra_path,
    nir=nir.spectra,
    nir_wavelength=nir.wavelength,
    tir=tir.spectra,
    tir_wavelength=tir.wavelength,
    depth_m=meta["depth_m"].values,
)
print(f"Saved spectra -> {spectra_path}  "
      f"({spectra_path.stat().st_size / 1e6:.0f} MB, down from ~180 MB of .bip)")

---
## 12. What we learned, and what to ask the client

### What the data is, in one table

| Thing | Reality |
|-------|---------|
| Holes available | ~1,950 packages (~2.3 TB) on THREDDS; 1,897 queryable live via the API |
| Access | ~1,200 open file, 132 confidential, ~600 with no status recorded |
| Location data | Lat/long for every hole, plus azimuth / dip / RL, free from the API |
| One hole | ~26,000 readings, one every ~8 mm of core |
| Spectra per reading | 531 VNIR/SWIR bands (380–2500 nm) + 341 TIR bands (6000–14500 nm) |
| Pre-computed scalars | 151 columns, of which ~75 are raw absorption-feature geometry |
| Mineral labels | Software guesses (TSA), present for only ~65% of readings, never verified |
| Extras | Core tray photos, a stitched mosaic, laser surface profile, calibration files |

### The five things that will shape the build

1. **Query, don't hoard.** 2.3 TB won't fit anywhere convenient, and `nvcl-kit` means we
   don't need it to. Pull per hole over HTTP; download full packages only when we need the
   raw spectral cube. This is an architecture decision the data just made for us.
2. **`-3.402823e+38` is "missing".** Clean it on read, once, in the ETL layer. If this leaks
   into a model, every result is wrong and it will look plausible.
3. **A third of every hole has no mineral answer,** and the missingness is not random —
   it tracks broken and weathered rock. Decide explicitly how to handle it and document it.
4. **`INVALID`, `Aspectral`, `NotInLibrary` are not minerals.** They are the software's way
   of saying "don't know". They must be excluded or modelled separately.
5. **There are no labels.** The TSA minerals are a *starting point for validation*, not
   ground truth. The brief is explicit: patterns must be defensible against the logged geology.

### Questions to put to Tasman (GSWA) at the next meeting

The two libraries he recommended already answered a lot — `nvcl-kit` gave us hole locations,
collar/survey, and per-log downloads, so those come off the list. What's left:

1. **Which holes are in scope?** All 1,897, or a defined subset (a region, a commodity, a
   tectonic unit)? The 132 confidential ones — in or out?
2. **Which hole list is authoritative?** The catalog spreadsheet and the live API disagree
   (section 3). The extras look like petroleum wells and cuttings — should they be excluded?
3. **Where do the assays and petrophysics come from?** The brief says cross-check against
   lab chemistry on the same core, but the HyLogger package contains none and the NVCL API
   doesn't appear to serve it. Is that a WAMEX-by-A-number extraction, and is there a bulk
   route rather than one report at a time?
4. **`sTSAS` vs `uTSAS` — which library do we trust?** Both are present and they disagree.
   Is one GSWA's standard? Has the user library been curated for these holes?
5. **Is there any logged geology (lithology intervals) for these holes?** That's the only
   thing we could validate clusters against. Without it, "explainable against the logged
   geology" has nothing to stand on.
6. **What counts as an "anomaly" to a geoscientist?** Statistically odd is easy. Odd *and
   worth driving to* is the actual product. We need their definition.
7. **Is there a bulk/polite way to hit the API?** If we're pulling logs for hundreds of
   holes, we should know their rate limits before we find them the hard way.

### Next notebook

Pull 10–20 holes via `nvcl-kit` and check whether they are comparable to each other at all:
same wavelength grid? same reference library? same TSG version? That comparability check is
the real prerequisite for clustering across holes, and it's worth doing before any modelling
code gets written.